<a href="https://colab.research.google.com/github/serdararici/btk-datathon-2026/blob/main/notebooks/v5_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Datathon 2026 — v5: Advanced (Aggressive Feature Engineering + Stacking)
**Goal:** Break the 91 MSE barrier  
**Previous:** v4 → Kaggle MSE 91.26  
**Strategy:** Interaction features, polynomial terms, stacking ensemble  
**Author:** Serdar Arıcı | **Date:** June 2026

In [1]:
# ============================================================
# SECTION 0: INSTALL & SETUP
# ============================================================

!pip install catboost optuna -q

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

DATA_PATH    = '/content/drive/MyDrive/Colab Notebooks/btk-datathon-2026/'
DATASET_PATH = DATA_PATH + 'datasets/'
OUTPUT_PATH  = DATA_PATH + 'outputs/'

train = pd.read_csv(DATASET_PATH + 'train.csv', encoding='utf-8-sig')
test  = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')

print(f"Train: {train.shape}, Test: {test.shape}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 22.4 MB/s eta 0:00:00
Mounted at /content/drive
Train: (10000, 47), Test: (10000, 46)


In [2]:
# ============================================================
# SECTION 1: BUILD V4 FEATURE SET (baseline reference)
# ============================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer

train_fe = train.copy()
test_fe  = test.copy()

# ---- Numeric feature engineering (from v4) ----
def create_features(df):
    technical_cols = ['coding_score', 'problem_solving_score', 'data_structures_score',
                      'sql_score', 'machine_learning_score', 'backend_score',
                      'frontend_score', 'cloud_score', 'devops_score']
    df['avg_technical_score'] = df[technical_cols].mean(axis=1)

    soft_cols = ['communication_score', 'teamwork_score',
                 'leadership_score', 'presentation_score']
    df['avg_soft_skill_score'] = df[soft_cols].mean(axis=1)

    df['experience_score'] = (df['internship_count'] * 3 + df['real_client_project_count'] * 2 +
                              df['freelance_project_count'] * 1 + df['hackathon_count'] * 1)
    df['interview_success_rate'] = df['interviews_attended'] / (df['applications_sent'] + 1)
    df['hackathon_efficiency']   = df['hackathon_awards'] / (df['hackathon_count'] + 1)
    df['github_impact']          = df['github_repo_count'] * df['github_avg_stars'].fillna(0)
    df['total_project_count']    = df['real_client_project_count'] + df['freelance_project_count']
    return df

train_fe = create_features(train_fe)
test_fe  = create_features(test_fe)

# ---- NLP features (from v4) ----
def clean_text(text):
    if pd.isnull(text): return ""
    text = text.replace('İ', 'i').replace('I', 'ı').lower()
    text = re.sub(r'[^a-zçğıöşü\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

train_fe['feedback_clean'] = train_fe['mentor_feedback_text'].apply(clean_text)
test_fe['feedback_clean']  = test_fe['mentor_feedback_text'].apply(clean_text)

positive_words = ['dikkat','çekici','güçlü','başarı','başarılı','yetkin','yetenek','yetenekli',
    'uzmanlık','mükemmel','kanıtlıyor','sağlam','ileri','kayda','değer','avantaj','aranan',
    'kaliteli','üstün','parlak','etkileyici','olumlu','güzel','yüksek','kuvvetli','önemli']
negative_words = ['ancak','gereken','gerekiyor','gerekmekte','gerekebilir','gerekli','eksik',
    'zayıf','yetersiz','geliştirmesi','geliştirmek','çalışması','destek','gelişime','gelişmekte',
    'sorun','düşük','daha','fazla']
potential_words = ['potansiyel','potansiyeli','umut','verici','motivasyon','gelişim','çaba',
    'azim','azmi','hedef','hedefleri']

def count_words(text, word_list):
    return sum(1 for w in text.split() if w in word_list)

def create_sentiment(df):
    df['positive_count'] = df['feedback_clean'].apply(lambda x: count_words(x, positive_words))
    df['negative_count'] = df['feedback_clean'].apply(lambda x: count_words(x, negative_words))
    df['potential_count'] = df['feedback_clean'].apply(lambda x: count_words(x, potential_words))
    df['sentiment_score'] = df['positive_count'] - df['negative_count']
    df['sentiment_ratio'] = df['positive_count'] / (df['positive_count'] + df['negative_count'] + 1)
    return df

train_fe = create_sentiment(train_fe)
test_fe  = create_sentiment(test_fe)

print("v4 feature set built. Ready for baseline CV.")

v4 feature set built. Ready for baseline CV.


In [3]:
# ============================================================
# SECTION 2: PREPROCESSING + BASELINE CV (5-FOLD)
# ============================================================

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor

def preprocess(train_fe, test_fe):
    """Full preprocessing pipeline - returns train_final, test_final"""
    train_processed = train_fe.copy()
    test_processed  = test_fe.copy()

    # Imputation
    duration_median = train_processed.loc[train_processed['internship_count'] > 0, 'internship_duration_months'].median()
    for df in [train_processed, test_processed]:
        df.loc[(df['internship_count'] == 0) & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = 0
        df.loc[(df['internship_count'] > 0)  & (df['internship_duration_months'].isnull()), 'internship_duration_months'] = duration_median

    numeric_to_impute = ['english_exam_score', 'portfolio_score', 'github_avg_stars',
                         'open_source_contribution_count', 'linkedin_profile_score', 'hr_interview_score']
    medians = {col: train_processed[col].median() for col in numeric_to_impute}
    train_processed.fillna(medians, inplace=True)
    test_processed.fillna(medians, inplace=True)

    # Encoding
    tier_mapping = {'Tier 1': 4, 'Tier 2': 3, 'Tier 3': 2, 'Tier 4': 1}
    train_processed['university_tier'] = train_processed['university_tier'].map(tier_mapping)
    test_processed['university_tier']  = test_processed['university_tier'].map(tier_mapping)

    target_col = train_processed['career_success_score'].copy()
    train_processed = train_processed.drop(columns=['career_success_score'])

    ohe_cols = ['department', 'target_role']
    train_processed = pd.get_dummies(train_processed, columns=ohe_cols, dtype=int)
    test_processed  = pd.get_dummies(test_processed,  columns=ohe_cols, dtype=int)
    train_processed, test_processed = train_processed.align(test_processed, join='left', axis=1, fill_value=0)

    noise_cols = ['hobby', 'preferred_social_media_platform', 'student_id',
                  'mentor_feedback_text', 'feedback_clean']
    for col in noise_cols:
        if col in train_processed.columns: train_processed = train_processed.drop(columns=[col])
        if col in test_processed.columns:  test_processed  = test_processed.drop(columns=[col])

    train_processed = train_processed.astype(float)
    test_processed  = test_processed.astype(float)
    train_processed['career_success_score'] = target_col.reset_index(drop=True)

    return train_processed, test_processed

# Build TF-IDF (fit on train, applied to both)
turkish_stopwords = ['ve','ile','bir','bu','da','de','için','olan','olarak','çok','gibi',
    'kadar','sonra','daha','en','ya','veya','ama','ise','göre','her','ki','mi','ne','o','şu']

def add_tfidf(train_fe, test_fe, max_features=50):
    tfidf = TfidfVectorizer(max_features=max_features, stop_words=turkish_stopwords, ngram_range=(1,2))
    tfidf_train = tfidf.fit_transform(train_fe['feedback_clean'])
    tfidf_test  = tfidf.transform(test_fe['feedback_clean'])
    cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
    tr_df = pd.DataFrame(tfidf_train.toarray(), columns=cols, index=train_fe.index).reset_index(drop=True)
    te_df = pd.DataFrame(tfidf_test.toarray(),  columns=cols, index=test_fe.index).reset_index(drop=True)
    return tr_df, te_df

# Run pipeline
train_processed, test_processed = preprocess(train_fe, test_fe)
tfidf_train_df, tfidf_test_df = add_tfidf(train_fe, test_fe)

train_processed = train_processed.reset_index(drop=True)
test_processed  = test_processed.reset_index(drop=True)

target = train_processed['career_success_score'].copy()
train_X = pd.concat([train_processed.drop(columns=['career_success_score']), tfidf_train_df], axis=1)
test_X  = pd.concat([test_processed, tfidf_test_df], axis=1)

train_X = train_X.loc[:, ~train_X.columns.duplicated()].astype(float)
test_X  = test_X.loc[:, ~test_X.columns.duplicated()].astype(float)

print(f"train_X shape: {train_X.shape}")
print(f"test_X shape:  {test_X.shape}")

# ---- BASELINE 5-FOLD CV ----
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)

cv_scores = cross_val_score(model, train_X, target, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
baseline_cv = -cv_scores.mean()

print(f"\nBASELINE 5-Fold CV MSE: {baseline_cv:.4f}")
print(f"Fold scores: {-cv_scores}")

train_X shape: (10000, 120)
test_X shape:  (10000, 120)

BASELINE 5-Fold CV MSE: 83.3777
Fold scores: [85.72738403 88.85009836 77.40785363 78.22096539 86.68196317]


In [4]:
# ============================================================
# SECTION 3: DATA-DRIVEN INTERACTION FEATURES
# ============================================================

def add_interactions(df):
    """
    Interaction features chosen based on correlation analysis:
    technical_interview_score combined with technical skills shows
    the strongest improvement over individual correlations.
    """
    # Top interactions: technical_interview_score x technical skills
    df['interview_x_problem_solving'] = df['technical_interview_score'] * df['problem_solving_score']
    df['interview_x_cloud']           = df['technical_interview_score'] * df['cloud_score']
    df['interview_x_coding']          = df['technical_interview_score'] * df['coding_score']
    df['interview_x_devops']          = df['technical_interview_score'] * df['devops_score']
    df['interview_x_backend']         = df['technical_interview_score'] * df['backend_score']

    # project_quality interactions (smaller gain but still positive)
    df['pq_x_problem_solving'] = df['project_quality_score'] * df['problem_solving_score']
    df['pq_x_interview']       = df['project_quality_score'] * df['technical_interview_score']

    return df

train_fe2 = add_interactions(train_fe.copy())
test_fe2  = add_interactions(test_fe.copy())

print("Interaction features added!")

# ---- RE-RUN PIPELINE WITH NEW FEATURES ----
train_processed2, test_processed2 = preprocess(train_fe2, test_fe2)
tfidf_train_df2, tfidf_test_df2 = add_tfidf(train_fe2, test_fe2)

train_processed2 = train_processed2.reset_index(drop=True)
test_processed2  = test_processed2.reset_index(drop=True)

target2 = train_processed2['career_success_score'].copy()
train_X2 = pd.concat([train_processed2.drop(columns=['career_success_score']), tfidf_train_df2], axis=1)
test_X2  = pd.concat([test_processed2, tfidf_test_df2], axis=1)

train_X2 = train_X2.loc[:, ~train_X2.columns.duplicated()].astype(float)
test_X2  = test_X2.loc[:, ~test_X2.columns.duplicated()].astype(float)

print(f"train_X2 shape: {train_X2.shape}  (was {train_X.shape})")

# ---- CV WITH NEW FEATURES ----
model = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
cv_scores2 = cross_val_score(model, train_X2, target2, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
interaction_cv = -cv_scores2.mean()

print(f"\nBaseline CV MSE:     {baseline_cv:.4f}")
print(f"With interactions:  {interaction_cv:.4f}")
print(f"Improvement:        {baseline_cv - interaction_cv:.4f}")

Interaction features added!
train_X2 shape: (10000, 127)  (was (10000, 120))

Baseline CV MSE:     83.3777
With interactions:  83.3214
Improvement:        0.0562


In [5]:
# ============================================================
# SECTION 4: LOG TRANSFORM FOR SKEWED FEATURES
# ============================================================

def add_log_transforms(df):
    """
    Apply log1p transform to right-skewed count/star features.
    log1p(x) = log(1+x), handles zeros safely.
    """
    skewed_cols = ['github_avg_stars', 'github_repo_count', 'open_source_contribution_count',
                   'hackathon_count', 'certification_count', 'applications_sent']
    for col in skewed_cols:
        df[f'{col}_log'] = np.log1p(df[col].fillna(0))
    return df

train_fe3 = add_log_transforms(train_fe2.copy())
test_fe3  = add_log_transforms(test_fe2.copy())

print("Log transforms added!")

# ---- RE-RUN PIPELINE ----
train_processed3, test_processed3 = preprocess(train_fe3, test_fe3)
tfidf_train_df3, tfidf_test_df3 = add_tfidf(train_fe3, test_fe3)

train_processed3 = train_processed3.reset_index(drop=True)
test_processed3  = test_processed3.reset_index(drop=True)

target3 = train_processed3['career_success_score'].copy()
train_X3 = pd.concat([train_processed3.drop(columns=['career_success_score']), tfidf_train_df3], axis=1)
test_X3  = pd.concat([test_processed3, tfidf_test_df3], axis=1)

train_X3 = train_X3.loc[:, ~train_X3.columns.duplicated()].astype(float)
test_X3  = test_X3.loc[:, ~test_X3.columns.duplicated()].astype(float)

print(f"train_X3 shape: {train_X3.shape}  (was {train_X2.shape})")

# ---- CV ----
model = XGBRegressor(n_estimators=400, max_depth=5, learning_rate=0.05, random_state=42, n_jobs=-1)
cv_scores3 = cross_val_score(model, train_X3, target3, cv=kf, scoring='neg_mean_squared_error', n_jobs=-1)
log_cv = -cv_scores3.mean()

print(f"\nBaseline CV MSE:        {baseline_cv:.4f}")
print(f"With interactions:     {interaction_cv:.4f}")
print(f"With log transforms:   {log_cv:.4f}")
print(f"Total improvement:     {baseline_cv - log_cv:.4f}")

Log transforms added!
train_X3 shape: (10000, 133)  (was (10000, 127))

Baseline CV MSE:        83.3777
With interactions:     83.3214
With log transforms:   83.2684
Total improvement:     0.1092


In [6]:
# ============================================================
# SECTION 5: STACKING - GENERATE OUT-OF-FOLD PREDICTIONS
# ============================================================

from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold

# Use the best feature set (train_X3 with interactions + log)
X = train_X3.values
y = target3.values
X_test = test_X3.values

# Base models with v4 best params
base_models = {
    'xgb': XGBRegressor(n_estimators=538, learning_rate=0.0198, max_depth=6,
                        subsample=0.635, colsample_bytree=0.858, random_state=42, n_jobs=-1),
    'lgb': lgb.LGBMRegressor(n_estimators=499, learning_rate=0.0411, max_depth=6,
                             num_leaves=20, subsample=0.609, colsample_bytree=0.603,
                             random_state=42, n_jobs=-1, verbose=-1),
    'cat': CatBoostRegressor(iterations=394, learning_rate=0.0706, depth=5,
                             l2_leaf_reg=8.75, random_state=42, verbose=0)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Storage for OOF predictions (train) and test predictions
oof_preds = {name: np.zeros(len(X)) for name in base_models}
test_preds = {name: np.zeros(len(X_test)) for name in base_models}

print("Generating out-of-fold predictions...\n")

for name, model in base_models.items():
    test_fold_preds = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_vl = X[train_idx], X[val_idx]
        y_tr, y_vl = y[train_idx], y[val_idx]

        # Train on 4 folds
        model.fit(X_tr, y_tr)

        # Predict the held-out fold (OOF)
        oof_preds[name][val_idx] = model.predict(X_vl)

        # Predict test set (will average across folds)
        test_fold_preds.append(model.predict(X_test))

    # Average test predictions across all folds
    test_preds[name] = np.mean(test_fold_preds, axis=0)

    # Report OOF score for this model
    oof_mse = mean_squared_error(y, oof_preds[name])
    print(f"{name.upper()} OOF MSE: {oof_mse:.4f}")

print("\nOOF predictions ready for meta-model!")

Generating out-of-fold predictions...

XGB OOF MSE: 82.1927
LGB OOF MSE: 81.3510
CAT OOF MSE: 80.7633

OOF predictions ready for meta-model!


In [7]:
# ============================================================
# SECTION 6: META-MODEL (STACKING)
# ============================================================

from sklearn.linear_model import Ridge

# ---- Build meta-features: OOF predictions become input features ----
meta_X_train = np.column_stack([oof_preds['xgb'], oof_preds['lgb'], oof_preds['cat']])
meta_X_test  = np.column_stack([test_preds['xgb'], test_preds['lgb'], test_preds['cat']])

print(f"Meta features shape - Train: {meta_X_train.shape}, Test: {meta_X_test.shape}")
print(f"Sample meta features (first 3 rows):")
print(meta_X_train[:3])

# ---- Meta-model: Ridge regression ----
# Ridge is simple and works well for combining a few correlated predictions
meta_model = Ridge(alpha=1.0)

# Cross-validate the meta-model on OOF predictions
meta_cv_scores = cross_val_score(meta_model, meta_X_train, y, cv=kf,
                                  scoring='neg_mean_squared_error')
meta_cv = -meta_cv_scores.mean()

print(f"\nStacking (Ridge meta-model) CV MSE: {meta_cv:.4f}")

# Compare with simple average
simple_avg_oof = (oof_preds['xgb'] + oof_preds['lgb'] + oof_preds['cat']) / 3
simple_avg_mse = mean_squared_error(y, simple_avg_oof)
print(f"Simple average OOF MSE:              {simple_avg_mse:.4f}")

# Fit meta-model on full OOF data, check learned weights
meta_model.fit(meta_X_train, y)
print(f"\nMeta-model weights: XGB={meta_model.coef_[0]:.3f}, LGB={meta_model.coef_[1]:.3f}, CAT={meta_model.coef_[2]:.3f}")
print(f"Meta-model intercept: {meta_model.intercept_:.3f}")

Meta features shape - Train: (10000, 3), Test: (10000, 3)
Sample meta features (first 3 rows):
[[89.73566437 85.08912596 86.26185892]
 [60.1901207  60.00005187 59.51583533]
 [87.82361603 84.67686163 85.57979675]]

Stacking (Ridge meta-model) CV MSE: 80.0226
Simple average OOF MSE:              80.2701

Meta-model weights: XGB=0.079, LGB=0.390, CAT=0.557
Meta-model intercept: -1.981


In [8]:
# ============================================================
# SECTION 7: FINAL STACKING SUBMISSION
# ============================================================

# Predict using meta-model on test meta-features
final_preds = meta_model.predict(meta_X_test)
final_preds = np.clip(final_preds, 0, 100)

# Build submission
test_original = pd.read_csv(DATASET_PATH + 'test_x.csv', encoding='utf-8-sig')
submission_stack = pd.DataFrame({
    'student_id': test_original['student_id'],
    'career_success_score': final_preds
})
submission_stack.to_csv(OUTPUT_PATH + 'submission_v5_stacking.csv', index=False)

print("Stacking submission created!")
print(submission_stack.head())
print(f"\nPrediction range: {final_preds.min():.2f} to {final_preds.max():.2f}")
print(f"Prediction mean:  {final_preds.mean():.2f}")

Stacking submission created!
   student_id  career_success_score
0  STU_010001             58.272600
1  STU_010002             74.786523
2  STU_010003             73.325124
3  STU_010004             97.812141
4  STU_010005             77.960921

Prediction range: 34.35 to 100.00
Prediction mean:  76.14


## v5 Stacking Results (Stage 1)

### New Features Added
- 7 interaction features (technical_interview_score × technical skills)
  → data-driven, chosen via correlation analysis
- 6 log1p transforms (github_avg_stars, github_repo_count, etc.)
  → reduces impact of outliers, correlation 0.151 → 0.184

### Stacking (Ridge meta-model)
- Base models: XGBoost, LightGBM, CatBoost (v4 tuned params)
- OOF MSE: XGB=82.19, LGB=81.35, CAT=80.76
- Learned meta-weights: XGB=0.079, LGB=0.390, CAT=0.557
- Stacking CV MSE: 80.02 (vs simple average 80.27)

### Kaggle Progress
| Version | Description              | Kaggle    |
|---------|---------------------------|-----------|
| V1      | Baseline                  | 99.089303 |
| V2      | Feature Engineering       | 98.827308 |
| V3      | NLP Features              | 97.334411 |
| V4      | Ensemble + Tuning          | 91.264980 |
| V5      | Stacking + interactions    | 90.644575 |

### Next Steps
- Enrich meta-features with original top features
- Try non-linear meta-model
- Final: extended Optuna tuning (5-fold, 50+ trials) - run once at the end